# The Wave Equation — Explicit Time Stepping and Energy Conservation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/wave.ipynb)

$\partial_{tt} u = c^2 \Delta u$ integrated with central differences. Being
second-order in time, the scheme needs two starting levels — the first step is
built from the initial velocity, and every later step reuses one factorised
operator.

The notebook then measures what a time integrator is really judged on: the
undamped wave equation conserves mechanical energy exactly, so the drift in
$E = \tfrac12 v^\top M v + \tfrac12 u^\top A u$ is a direct read-out of
scheme quality.

⏱️ *A couple of minutes on Colab's free CPU runtime — most of it in
rendering the movie rather than in the solve.*

Docs: [Wave Equation](https://docs.tensor-mesh.com/example_gallery/wave.html) · Source: [`examples/wave/wave.py`](https://github.com/camlab-ethz/TensorMesh/blob/main/examples/wave/wave.py)

In [ ]:
# Install TensorMesh (skipped automatically if it is already available, e.g. a local dev setup).
# The apt line provides the OpenGL utility library that gmsh -- TensorMesh's mesh generator --
# needs at import time; it is a no-op where the library is already present.
# ffmpeg backs matplotlib's movie writer, which turns the time series into mp4.
import importlib.util
if importlib.util.find_spec("tensormesh") is None:
    !apt-get -qq install -y libglu1-mesa ffmpeg > /dev/null 2>&1 || true
    %pip install -q tensormesh-fem==0.2.0

import contextlib
import os


@contextlib.contextmanager
def quiet():
    """Hide gmsh's meshing log, which is written below Python's stdout.
    Drop the ``with quiet():`` wrapper anywhere to see what the mesher is doing."""
    with open(os.devnull, "w") as null:
        saved = os.dup(1)
        os.dup2(null.fileno(), 1)
        try:
            yield
        finally:
            os.dup2(saved, 1)
            os.close(saved)

In [ ]:
import warnings

import matplotlib.pyplot as plt
import torch

from tensormesh import Condenser, ElementAssembler, Mesh
from tensormesh.dataset import WaveMultiFrequency
from tensormesh.sparse import SparseMatrix

# Raised from inside PyTorch during assembly; nothing to act on here.
warnings.filterwarnings("ignore", message=".*torch.meshgrid.*")


class Stiffness(ElementAssembler):
    def forward(self, gradu, gradv):
        return gradu @ gradv


class MassMatrix(ElementAssembler):
    def forward(self, u, v):
        return u * v


def scale(mat, s):
    """Scale a SparseMatrix, preserving its type."""
    return SparseMatrix(mat.edata * s, mat.row, mat.col, mat.shape)

## Time stepping

In [ ]:
torch.manual_seed(123456)

CHARA_LENGTH = 0.025
N_STEPS = 90
DT, C = 1e-3, 2.0

with quiet():
    mesh = Mesh.gen_rectangle(chara_length=CHARA_LENGTH)
dataset = WaveMultiFrequency(K=16, c=C)
print(f"mesh: {mesh.n_points} nodes")

M = MassMatrix.from_mesh(mesh, quadrature_order=2)()
A = scale(Stiffness.from_mesh(mesh, quadrature_order=2)(), C * C)
condenser = Condenser(mesh.boundary_mask)

# Central differences: M (U^{n+1} - 2U^n + U^{n-1}) = -dt^2 A U^n.
u0 = dataset.initial_condition(mesh.points)
v0 = torch.zeros_like(u0)
Us = [u0]

# First step uses the initial velocity, so it needs its own operator.
K = scale(M, 2.0)
F = -(DT * DT) * (A @ u0) + 2.0 * (M @ u0) + (2.0 * DT) * (M @ v0)
K_, F_ = condenser(K, F)
Us.append(condenser.recover(K_.solve(F_)))
M_ = scale(K_, 0.5)          # K_ = 2 M_, so the steady-state operator is K_ / 2

for _ in range(N_STEPS - 2):
    U1, U2 = Us[-2:]
    F = 2.0 * (M @ U2) - (M @ U1) - (DT * DT) * (A @ U2)
    Us.append(condenser.recover(M_.solve(condenser.condense_rhs(F))))

print(f"{len(Us)} time levels computed")

## Energy conservation

Kinetic and potential energy trade back and forth; the total should stay flat.
Central differences are symplectic, so the drift stays bounded rather than
growing with the number of steps.

In [ ]:
# The undamped wave equation conserves E = 1/2 v^T M v + 1/2 u^T A u (A carries c^2).
# How well a scheme preserves it is the real test of a time integrator.
def quad(mat, x):
    return 0.5 * torch.dot(x, mat @ x)


times, ke, pe = [], [], []
for i, Ui in enumerate(Us):
    if i == 0:
        v = v0
    elif i == len(Us) - 1:
        v = (Us[i] - Us[i - 1]) / DT
    else:
        v = (Us[i + 1] - Us[i - 1]) / (2.0 * DT)
    times.append(DT * i)
    ke.append(quad(M, v).item())
    pe.append(quad(A, Ui).item())

ke, pe = torch.tensor(ke), torch.tensor(pe)
E = ke + pe
drift = ((E[-1] - E[0]) / E[0]).item()
print(f"E_0 = {E[0]:.6e},  E_final = {E[-1]:.6e},  relative drift = {drift:+.2e}")

fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(times, ke, label=r"kinetic  $\frac{1}{2} v^\top M v$", color="#2980b9")
ax.plot(times, pe, label=r"potential  $\frac{1}{2} u^\top A u$", color="#27ae60")
ax.plot(times, E, label="total", color="#c0392b", lw=2)
ax.set_xlabel("time"); ax.set_ylabel("energy")
ax.set_title(f"Wave energy (relative drift {drift:+.1e})")
ax.grid(alpha=0.3); ax.legend(loc="center right")
fig.tight_layout()
plt.show()

## Animation

In [ ]:
Us_exact = [dataset.solution(mesh.points, DT * i) for i in range(len(Us))]

mesh.plot(
    {"FEM solution": Us, "Analytical solution": Us_exact},
    save_path="wave.mp4", dt=DT, show_mesh=False, linewidth=0.1, linecolor="black",
)
from IPython.display import Video
Video("wave.mp4", embed=True, width=780)